# KARMA Experiments
**K-order Approximation via Markov chains for Retrospective Attribution**

This notebook runs the full KARMA explainability pipeline on any supported time-series dataset using a pre-trained LSTM or TCN oracle.

**Pipeline stages**
1. Train oracle (LSTM / TCN)
2. KARMA — K* selection + transition kernel + causal DAG (Pillars 2 & 3)
3. TimeSHAP — temporal coalition pruning + KernelSHAP (baseline comparison)
4. Visualisation — five explanation levels

**Supported datasets:** `etth1` · `ettm2` · `weather` · `exchange_rate` · `beijing` · `electricity` · `web_traffic`

---
> **Runtime:** GPU (T4 or better recommended). Go to *Runtime → Change runtime type → T4 GPU*.

## 0 · Environment setup

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU:", result.stdout.strip())
else:
    print("No GPU detected — training will run on CPU (slower).")

In [ ]:
# ── Clone repository ───────────────────────────────────────────────────────────
import os

REPO_URL  = "https://github.com/AmTuTi1999/KARMA-.git"
REPO_ROOT = "/content/KARMA-"

if not os.path.isdir(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
else:
    print("Repo already cloned — pulling latest.")
    !git -C {REPO_ROOT} pull --ff-only

%cd {REPO_ROOT}

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
# torch / numpy / pandas / sklearn / matplotlib are pre-installed in Colab.
# Install the extras needed by KARMA.
!pip install -q \
    pyyaml \
    tqdm \
    tigramite \
    timeshap \
    shap==0.40.0 \
    statsmodels \
    mlflow

# Make repo packages importable without installing
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Done.")

## 1 · Data

In [ ]:
# ── Pull data from Git LFS ─────────────────────────────────────────────────────
# Data (raw CSVs + pre-processed .npy arrays) and model checkpoints are stored
# in Git LFS.  A single `git lfs pull` fetches everything.

!apt-get install -y -q git-lfs
!git lfs install --skip-repo
!git lfs pull

import pathlib
npy_count = len(list(pathlib.Path(f"{REPO_ROOT}/data").rglob("*.npy")))
print(f"\nData ready — {npy_count} .npy files found.")

In [ ]:
# ── Verify a dataset ───────────────────────────────────────────────────────────
import numpy as np

ds = "etth1"   # quick sanity-check on a small dataset
data_path = pathlib.Path(f"{REPO_ROOT}/data/generated/{ds}")

for split in ("train", "val", "test"):
    X = np.load(data_path / f"X_{split}.npy")
    y = np.load(data_path / f"y_{split}.npy")
    print(f"  {split}: X={X.shape}  y={y.shape}  dtype={X.dtype}")

## 2 · Configuration

Choose your dataset and model type, then run the path-patching cell so all configs point to Colab paths.

In [ ]:
# ── Experiment knobs ───────────────────────────────────────────────────────────
DATASET    = "etth1"      # etth1 | ettm2 | weather | exchange_rate
                          # beijing | electricity | web_traffic
MODEL_TYPE = "lstm"       # lstm | tcn
EPOCHS     = 30           # training epochs (reduce for a quick test)
SEED       = 42

In [ ]:
# ── Patch dataset configs: replace dev-container paths with Colab paths ────────
import yaml

OLD_PREFIX = "/workspaces/KARMA-"
cfg_dir    = pathlib.Path(f"{REPO_ROOT}/configs/datasets")

for cfg_path in sorted(cfg_dir.glob("*.yaml")):
    text = cfg_path.read_text()
    if OLD_PREFIX in text:
        cfg_path.write_text(text.replace(OLD_PREFIX, REPO_ROOT))
        print(f"  patched {cfg_path.name}")

# Verify the active dataset config
cfg = yaml.safe_load((cfg_dir / f"{DATASET}.yaml").read_text())
print(f"\n{DATASET} config:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

## 3 · Train oracle (LSTM / TCN)

Trains the neural-network forecaster that KARMA will explain.
Skip this cell if you already have a checkpoint under `outputs/checkpoints/`.

In [ ]:
import torch
from pipeline.training_pipeline import train_model

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

pipeline_obj, test_results = train_model(
    dataset_name=DATASET,
    model_type=MODEL_TYPE,
    epochs=EPOCHS,
    batch_size=64,
    seed=SEED,
)

print("\nTest results:")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 4 · KARMA pipeline

**Pillar 2** — finds the minimal lag order K* and baseline b* such that the K-order Markov surrogate matches the oracle within tolerance ε.  
**Pillar 3** — estimates the transition kernel T̂(·|h) via Monte Carlo and recovers the causal DAG.

In [ ]:
from pipeline.karma_pipeline import run_karma

karma_results = run_karma(
    dataset_name=DATASET,
    model_type=MODEL_TYPE,
    N=3,          # bins per variable
    eps=0.05,     # Δ^pred tolerance
    lam=0.025,    # edge-trimming threshold
    M=64,         # MC draws per history
    K_max=4,
    seed=SEED,
    mega_batch=True,   # fuse all H*M windows into one GPU call
    verbose=True,
)

p2 = karma_results["pillar2"]
p3 = karma_results["pillar3"]
print(f"\n── Pillar 2 summary ──")
print(f"  K*            = {p2['K_star']}")
print(f"  b*            = {p2['b_star_name']}")
print(f"  Compression   = {p2['compression_ratio']:.1f}×")
print(f"  Δ^pred [cert] = {p2['delta_pred']:.4f}")
print(f"\n── Pillar 3 summary ──")
print(f"  Retained edges: {len(p3['retained_edges'])}")

In [ ]:
# ── Variable importance table ──────────────────────────────────────────────────
import pandas as pd

vi  = p3["variable_importance"]
cfg = karma_results["config"]
var_names = yaml.safe_load((cfg_dir / f"{DATASET}.yaml").read_text()).get("var_names")
if var_names is None:
    var_names = [f"X^{d}" for d in range(cfg["D"])]

df_vi = pd.DataFrame({
    "variable": var_names,
    "Phi": vi["Phi"],
    "Phi_n": vi["Phi_n"],
}).sort_values("Phi_n", ascending=False).reset_index(drop=True)

print("Variable importance (normalised):")
display(df_vi.style.bar(subset=["Phi_n"], color="#4C72B0").format({"Phi": "{:.4f}", "Phi_n": "{:.4f}"}))

In [ ]:
# ── Retained causal edges ──────────────────────────────────────────────────────
df_edges = pd.DataFrame(p3["retained_edges"])
if not df_edges.empty:
    df_edges = df_edges[["src_name", "lag", "tgt_name", "rho"]].sort_values("rho", ascending=False)
    print(f"Causal edges (ρ ≥ {karma_results['config']['lam']}):")
    display(df_edges.style.bar(subset=["rho"], color="#55A868").format({"rho": "{:.4f}"}))
else:
    print("No edges retained (try lowering lam)")

## 5 · TimeSHAP baseline

Runs the TimeSHAP pipeline (KernelSHAP adapted for sequential models) for comparison with KARMA.

In [ ]:
from pipeline.timeshap_pipeline import run_timeshap

ts_results = run_timeshap(
    dataset_name=DATASET,
    model_type=MODEL_TYPE,
    eps=0.05,
    baseline_strategy="mean",
    n_samples=256,    # reduce for speed; 512 for publication quality
    n_explain=30,
    mode="feature",
    seed=SEED,
    verbose=True,
)

print(f"\nK_pruned = {ts_results['pruning']['K_pruned']}")

In [ ]:
# ── TimeSHAP variable importance ───────────────────────────────────────────────
ts_vi = ts_results["shap"]["variable_importance"]
df_ts = pd.DataFrame({
    "variable": var_names,
    "var_imp":   ts_vi["var_imp"],
    "var_imp_n": ts_vi["var_imp_n"],
}).sort_values("var_imp_n", ascending=False).reset_index(drop=True)

print("TimeSHAP variable importance:")
display(df_ts.style.bar(subset=["var_imp_n"], color="#C44E52").format({"var_imp": "{:.4f}", "var_imp_n": "{:.4f}"}))

In [ ]:
# ── Side-by-side comparison ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KARMA
ax = axes[0]
idx = np.argsort(vi["Phi_n"])[::-1]
ax.barh([var_names[i] for i in idx], [vi["Phi_n"][i] for i in idx], color="#4C72B0")
ax.set_xlabel("Normalised importance")
ax.set_title("KARMA — variable importance")
ax.invert_yaxis()

# TimeSHAP
ax = axes[1]
idx2 = np.argsort(ts_vi["var_imp_n"])[::-1]
ax.barh([var_names[i] for i in idx2], [ts_vi["var_imp_n"][i] for i in idx2], color="#C44E52")
ax.set_xlabel("Normalised importance")
ax.set_title("TimeSHAP — variable importance")
ax.invert_yaxis()

fig.suptitle(f"{DATASET.upper()} · {MODEL_TYPE.upper()}", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 6 · KARMA visualisation (Levels 1 – 5)

| Level | What it shows |
|-------|---------------|
| 1 | Global variable importance Φ̃ᵈ |
| 2 | Lag profiles φᵈₖ — which lags of each variable matter |
| 3 | Regime distance matrix TV(T̂(·|h), T̂(·|h′)) |
| 4 | Ranked TV edge contributions ρ(e) — causal DAG |
| 5 | Uncertainty dashboard — aleatoric / epistemic / noise floor |

In [ ]:
from pipeline.visualization import visualize
import pathlib

results_path = pathlib.Path(f"{REPO_ROOT}/results/{DATASET}/{MODEL_TYPE}/karma_results.json")
figures_dir  = pathlib.Path(f"{REPO_ROOT}/figures/{DATASET}/{MODEL_TYPE}")
figures_dir.mkdir(parents=True, exist_ok=True)

# Auto-detect target variable (last variable is often the forecast target)
target_var = var_names[-1]

figs = visualize(
    results_path=results_path,
    figures_dir=figures_dir,
    target_var=target_var,
    lam=karma_results["config"]["lam"],
    fmt="png",    # pdf works too, but png renders inline in Colab
    verbose=True,
)

print(f"\nSaved {len(figs)} figures to {figures_dir}")

In [ ]:
# ── Display figures inline ─────────────────────────────────────────────────────
from IPython.display import display as ipy_display
import matplotlib.pyplot as plt

LEVEL_LABELS = {
    "karma_level1_importance": "Level 1 — Variable importance",
    "karma_level2_lagprofiles": "Level 2 — Lag profiles",
    "karma_level3_regime": "Level 3 — Regime distance matrix",
    "karma_level4_causal": "Level 4 — Causal DAG edges",
    "karma_level5_uncertainty": "Level 5 — Uncertainty dashboard",
    "karma_overview": "Overview",
}

for key, label in LEVEL_LABELS.items():
    if key in figs:
        fig = figs[key]
        fig.suptitle(label, fontsize=12, fontweight="bold", y=1.01)
        plt.figure(fig.number)
        plt.show()

## 7 · Multi-dataset sweep

Run KARMA across all datasets and collect summary metrics. This cell may take **30 – 90 minutes** depending on dataset sizes and GPU speed.

Set `RUN_SWEEP = True` to execute.

In [ ]:
RUN_SWEEP = False   # set True to run all datasets

SWEEP_DATASETS = ["etth1", "ettm2", "exchange_rate", "weather"]
SWEEP_MODELS   = ["lstm", "tcn"]

if RUN_SWEEP:
    from pipeline.training_pipeline import train_model
    from pipeline.karma_pipeline    import run_karma

    summary_rows = []
    for ds in SWEEP_DATASETS:
        # Patch this dataset's config path
        cfg_path = cfg_dir / f"{ds}.yaml"
        text = cfg_path.read_text()
        if OLD_PREFIX in text:
            cfg_path.write_text(text.replace(OLD_PREFIX, REPO_ROOT))

        for mt in SWEEP_MODELS:
            print(f"\n{'='*55}")
            print(f"  Dataset={ds}  Model={mt}")
            print("="*55)

            try:
                train_model(dataset_name=ds, model_type=mt,
                            epochs=EPOCHS, seed=SEED)
            except Exception as e:
                print(f"  [SKIP train] {e}")
                continue

            try:
                r = run_karma(dataset_name=ds, model_type=mt,
                              eps=0.05, lam=0.025, M=64,
                              mega_batch=True, seed=SEED, verbose=False)
                p2 = r["pillar2"]
                p3 = r["pillar3"]
                summary_rows.append({
                    "dataset":     ds,
                    "model":       mt,
                    "K_star":      p2["K_star"],
                    "compression": p2["compression_ratio"],
                    "delta_pred":  p2["delta_pred"],
                    "n_edges":     len(p3["retained_edges"]),
                    "pool_cov":    p3["pool_stats"]["pool_coverage"],
                })
            except Exception as e:
                print(f"  [SKIP KARMA] {e}")

    df_summary = pd.DataFrame(summary_rows)
    print("\n=== Sweep summary ===")
    display(df_summary)

## 8 · Save results to Google Drive (optional)

Mount Drive and copy the results and figures folders for persistence across sessions.

In [ ]:
SAVE_TO_DRIVE = False   # set True to save

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DIR = f"/content/drive/MyDrive/KARMA_results/{DATASET}_{MODEL_TYPE}"
    !mkdir -p {DRIVE_DIR}
    !cp -r {REPO_ROOT}/results/{DATASET}   {DRIVE_DIR}/results
    !cp -r {REPO_ROOT}/figures/{DATASET}   {DRIVE_DIR}/figures
    print(f"Saved to {DRIVE_DIR}")